In [1]:
########################################  Instalar librerias  ##########################################   
!pip install openpyxl # para cargar los textos y guardar los resultados
########################################  Instalar librerias  ##########################################   

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 21.0 MB/s eta 0:00:00


In [4]:
##########################################  Importaciones  ##########################################
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
import torch
import gc
import unittest
import os

import time #borrar
##########################################  Importaciones  ##########################################
##########################################  Cargar excell  ##########################################
def cargar_textos(ruta_archivo):
    """
    Cargar el archivo excel de ruta_archivo, descarta cualquier valor nulo
    y devuelve la primera columna del excel en forma de lista. Si hay cualquier error se para la ejcucion.
    """
    try:
        df = pd.read_excel(ruta_archivo, engine='openpyxl')
        return df.iloc[0:, 0].dropna().tolist()
    
    except FileNotFoundError as e:
        print(f"Archivo {ruta_archivo} no encontrado: {e}")
        raise
        
    except Exception as e:
        print(f"Error al cargar o procesar el archivo {ruta_archivo}: {e}")
        raise
##########################################  Cargar excell  ##########################################
##########################################  Resultados a excell  ##########################################   
def guardar_resultados(resultados, nombre_columna, ruta):
    """
    Se guardan los resultados en el archivo resultados.xlsx, sino está creado se crea. 
    resultados se guarda en la primera columna libre que haya en resultados.xlxs, si nombre_columna 
    ya existe se sobreescribiran resultados en ella.
    """
    try:
        df = pd.read_excel(ruta, engine='openpyxl')
        
    except FileNotFoundError:
        df = pd.DataFrame()
    serie_resultados = pd.Series(resultados)
    
    if nombre_columna in df.columns:
        df[nombre_columna] = serie_resultados
    else:
        df[nombre_columna] = serie_resultados

    with pd.ExcelWriter(ruta, engine='openpyxl', mode='w') as writer:
        df.to_excel(writer, index=False)
##########################################  Resultados a excell  ##########################################   
##########################################  Limpiar memoria  ##########################################   
def liberar_memoria_gpu():
    try:
        gc.collect()
        torch.cuda.empty_cache()  
        
    except Exception as e:
        raise Exception(f"Error al intentar liberar memoria GPU: {e}") 
##########################################  Limpiar memoria  ##########################################
##########################################  Cargar modelo  ##########################################   
def cargar_modelo(model_name):
    """
    Carga y descarga de los modelos. Primero carga el tokenizador, después el modelo model_name si el 
    modelo no esta descargado hay que descomentar la linea del token_id e introducir uno valido. Si el modelo no 
    esta configurado salta un error y se para la ejecucion, si ocurre un error desconocido tambien para la ejcucion.
    """
    token_id = ""
    torch.manual_seed(42)
    print("Hora de carga: ",time.strftime("%X")) #borrar
    
    if(model_name == "microsoft/Phi-3-medium-128k-instruct"):
        print(f"Cargando {model_name} ...")
        tokenizer = AutoTokenizer.from_pretrained(model_name
                                                  #,token=token_id
                                                 )
        model = AutoModelForCausalLM.from_pretrained(
                                                    model_name,
                                                    #token=token_id,
                                                    torch_dtype=torch.bfloat16,
                                                    device_map="auto",
                                                    trust_remote_code=True)
        model.config.attn_implementation = 'eager'
        
    elif(model_name=="meta-llama/Meta-Llama-3-8B-Instruct"):
        print(f"Cargando {model_name} ...")
        tokenizer = AutoTokenizer.from_pretrained(model_name
                                                  #,token=token_id
                                                 )
        model = AutoModelForCausalLM.from_pretrained(
                                                    model_name,
                                                    #token=token_id,
                                                    torch_dtype=torch.bfloat16,
                                                    device_map="auto")
        
    else:
        raise ValueError(f"El modelo \"{model_name}\" no tiene implementada su carga.")
    
    return model, tokenizer

##########################################  Cargar modelo  ##########################################
##########################################  Generar respuestas  #######################################  
def generar_respuestas(model_name, textos, prompt, gen_config, ruta):
    """
    Primero se carga el modelo y tokenizador de model_name. Se configura el eos y el pad token. 
    Se recorre los textos de los que se desea generar una respuesta al prompt
    con la decodificacion gen_config. Por ultimo guarda las respuestas en un Excel.
    """
    print(f"Configurando {model_name} ...")
    model,tokenizer = cargar_modelo(model_name)
            
    resultados=[]
    try:
        if(model_name == "meta-llama/Meta-Llama-3-8B-Instruct"):
            
            terminators = [
                            tokenizer.eos_token_id, 
                            tokenizer.convert_tokens_to_ids("<|eot_id|>")
                          ]
            
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
                tokenizer.pad_token_id = tokenizer.eos_token_id  

            gen_config.eos_token_id = terminators
            gen_config.pad_token_id = tokenizer.pad_token_id

            for texto in textos:
                messages  = [
                             {"role": "system", "content": prompt},
                             {"role": "user", "content": texto},
                            ]
                
                gen_config.max_new_tokens=len(tokenizer.tokenize(texto)) + 5
                
                inputs = tokenizer.apply_chat_template(
                            messages,
                            add_generation_prompt=True,
                            return_tensors="pt").to("cuda")
                
                outputs = model.generate(
                            inputs,
                            generation_config=gen_config)

                response = outputs[0][inputs.shape[-1]:]
                res = tokenizer.decode(response, skip_special_tokens=True)
                resultados.append(res)
            guardar_resultados(resultados,model_name, ruta)
        
        else:
            
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
                tokenizer.pad_token_id = tokenizer.eos_token_id

            gen_config.pad_token_id = tokenizer.pad_token_id

            for texto in textos:
                messages = [
                            {"role": "user", "content": prompt},
                            {"role": "assistant", "content": ""},
                            {"role": "user", "content": texto}
                            ]
                
                gen_config.max_new_tokens=len(tokenizer.tokenize(texto)) + 5
                
                inputs = tokenizer.apply_chat_template(
                             messages,
                             add_generation_prompt=True,
                             return_tensors="pt").to("cuda")
                
                outputs = model.generate(
                            inputs,
                            generation_config=gen_config)
                
                response = outputs[0][inputs.shape[-1]:]
                res = tokenizer.decode(response, skip_special_tokens=True)
                resultados.append(res) 
    
            guardar_resultados(resultados,model_name,ruta)

    except Exception as e:
        print(f"Error: {e}")
              
    print("----------------------FIN--------------------------Hora: ",time.strftime("%X"))
##########################################  Generar respuestas  #######################################  

In [3]:
##########################################  Ejecucion  ##########################################   
def main():
    textos = cargar_textos("/home/jovyan/data/textos.xlsx")
    prompts =[
        """Sustituye las etiquetas <mask> del siguiente texto por conectores discursivos espaciales o temporales.
Devuelve en tu respuesta el texto original completo con los conectores, sin añadir más información."""]
    
    gen_config = GenerationConfig(num_beams=1, 
                                  do_sample=False)
    
    models = ["meta-llama/Meta-Llama-3-8B-Instruct",  "microsoft/Phi-3-medium-128k-instruct"
              
             ]
    
    i=1
    for prompt in prompts:
        ruta=f"/home/jovyan/data/resultados_Prompt_{i}.xlsx"
        i=i+1
        for model_name in models:
            generar_respuestas(model_name, textos, prompt, gen_config,ruta)
            liberar_memoria_gpu()

if __name__ == "__main__":
    main()
##########################################  Ejecucion  ##########################################   

Configurando meta-llama/Meta-Llama-3-8B-Instruct ...
Hora de carga:  12:21:51
Cargando meta-llama/Meta-Llama-3-8B-Instruct ...


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Mientras nos encontrábamos en mi hamaca cuando miré los calcetines que traía puestos, unos que él me había prestado, y le dije las palabras que me atemorizaba tanto pronunciar: “Te amo”. La hamaca se mecía, los grillos cantaban. “Gracias, pero yo todavía no te amo”, dijo. Señaló mis pies. “Esos no me quedan. ¿Los quieres?”. Sin embargo, Sentí como si la hamaca se hubiera volcado y me hubiera lanzado con violencia; no me amaba, aquí terminaba todo. Pero años más tarde, mientras seguimos recostándonos en mi hamaca y yo sigo poniéndome esos calcetines. Tenía razón: son demasiado pequeños para sus pies.
“Fábula de las ranas pidiendo al rey”_x000D_Las ranas vivían en el caos y la anarquía, y estaban cansadas de esta situación. Así que entonces mandaron una delegación para pedirle a Zeus, el rey de los dioses, que les enviara un rey._x000D_Zeus, atendiendo su petición, les envió un grueso leño a su charca._x000D_Las ranas se asustaron con el ruido que hizo el leño al caer, y se escondieron e

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attenton` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

You are not running the flash-attention implementation, expect numerical differences.


"En ese momento, Nos encontrábamos en mi hamaca cuando miré los calcetines que traía puestos, unos que él me había prestado, y le dije las palabras que me atemorizaba tanto pronunciar: “Te amo”. La hamaca se mecía, los grillos cantaban. “Gracias, pero yo todavía no te amo”, dijo. Señaló mis pies. “Esos no me quedan. ¿Los quieres?”. Inmediatamente, Sentí como si la hamaca se hubiera volcado y me hubiera lanzado con violencia; no me amaba, aquí terminaba todo. Pero años más tarde, seguimos recostándonos en mi hamaca y yo sigo poniéndome esos calcetines. Tenía razón: son demasiado pequeños para sus pies.
“Fábula de las ranas pidiendo al rey”_x000D_Las ranas vivían en el caos y la anarquía, y estaban cansadas de esta situación. Así que <entonces> mandaron una delegación para pedirle a Zeus, el rey de los dioses, que les enviara un rey._x000D_Zeus, atendiendo su petición, les envió un grueso leño a su charca._x000D_Las ranas se asustaron con el ruido que hizo el leño al caer, y se escondier

In [2]:
class TestCargarExcel(unittest.TestCase):

    def setUp(self):
        # Crear archivos Excel de prueba
        data = {'Textos': ['Texto 1', 'Texto 2', 'Texto 3']}
        df = pd.DataFrame(data)
        df.to_excel('archivo_correcto.xlsx', index=False)

        data_textos_vacio = {'Textos': []}
        df_textos_vacio = pd.DataFrame(data_textos_vacio)
        df_textos_vacio.to_excel('archivo_textos_vacio.xlsx', index=False)
        
        data_con_filas_vacias = {'Textos': ['Texto 1', None, 'Texto 3']}
        df_con_filas_vacias = pd.DataFrame(data_con_filas_vacias)
        df_con_filas_vacias.to_excel('archivo_con_filas_vacias.xlsx', index=False)

        # Crear un archivo que no sea Excel
        with open('archivo_no_excel.txt', 'w') as f:
            f.write('Este no es un archivo Excel.')

    def tearDown(self):
        # Eliminar los archivos de prueba después de cada prueba
        files_to_remove = [
            'archivo_correcto.xlsx',
            'archivo_textos_vacio.xlsx',
            'archivo_con_filas_vacias.xlsx',
            'archivo_no_excel.txt'
        ]
        for file in files_to_remove:
            if os.path.exists(file):
                os.remove(file)

    def test_cargar_textos_excel_ruta_vacia(self):
        with self.assertRaises(FileNotFoundError):
            cargar_textos('')

    def test_cargar_textos_excel_archivo_no_existe(self):
        with self.assertRaises(FileNotFoundError):
            cargar_textos('archivo_no_existe.xlsx')

    def test_cargar_textos_excel_no_excel(self):
        with self.assertRaises(Exception):
            cargar_textos('archivo_no_excel.txt')

    def test_cargar_textos_excel_columna_vacia(self):
        textos = cargar_textos('archivo_textos_vacio.xlsx')
        self.assertEqual(textos, [])

    def test_cargar_textos_excel_filas_vacias(self):
        textos = cargar_textos('archivo_con_filas_vacias.xlsx')
        self.assertEqual(textos, ['Texto 1', 'Texto 3'])

    def test_cargar_textos_excel_correcto(self):
        textos = cargar_textos('archivo_correcto.xlsx')
        self.assertIsInstance(textos, list)
        self.assertEqual(len(textos), 3)
        self.assertEqual(textos, ['Texto 1', 'Texto 2', 'Texto 3'])
if __name__ == '__main__':
    unittest.main(argv=[''], exit=False) 

.....

Archivo archivo_no_existe.xlsx no encontrado: [Errno 2] No such file or directory: 'archivo_no_existe.xlsx'
Error al cargar o procesar el archivo archivo_no_excel.txt: File is not a zip file


.
----------------------------------------------------------------------
Ran 6 tests in 0.361s

OK


Archivo  no encontrado: [Errno 2] No such file or directory: ''


In [3]:
class TestGuardarResultados(unittest.TestCase):
    def setUp(self):
        # Crear archivos Excel de prueba
        data = {'Textos': ['Texto 1', 'Texto 2', 'Texto 3']}
        df = pd.DataFrame(data)
        df.to_excel('archivo_correcto.xlsx', index=False)

        data_textos_vacio = {'Textos': []}
        df_textos_vacio = pd.DataFrame(data_textos_vacio)
        df_textos_vacio.to_excel('archivo_textos_vacio.xlsx', index=False)
        
        data_con_filas_vacias = {'Textos': ['Texto 1', None, 'Texto 3']}
        df_con_filas_vacias = pd.DataFrame(data_con_filas_vacias)
        df_con_filas_vacias.to_excel('archivo_con_filas_vacias.xlsx', index=False)

        # Crear un archivo que no sea Excel
        with open('archivo_no_excel.txt', 'w') as f:
            f.write('Este no es un archivo Excel.')

    def tearDown(self):
        # Eliminar los archivos de prueba después de cada prueba
        files_to_remove = [
            'archivo_correcto.xlsx',
            'archivo_textos_vacio.xlsx',
            'archivo_con_filas_vacias.xlsx',
            'archivo_no_excel.txt'
        ]
        for file in files_to_remove:
            if os.path.exists(file):
                os.remove(file)

    def test_cargar_textos_excel_ruta_vacia(self):
        with self.assertRaises(FileNotFoundError):
            cargar_textos('')

    def test_cargar_textos_excel_archivo_no_existe(self):
        with self.assertRaises(FileNotFoundError):
            cargar_textos('archivo_no_existe.xlsx')

    def test_cargar_textos_excel_no_excel(self):
        with self.assertRaises(Exception):
            cargar_textos('archivo_no_excel.txt')

    def test_cargar_textos_excel_columna_vacia(self):
        textos = cargar_textos('archivo_textos_vacio.xlsx')
        self.assertEqual(textos, [])

    def test_cargar_textos_excel_filas_vacias(self):
        textos = cargar_textos('archivo_con_filas_vacias.xlsx')
        self.assertEqual(textos, ['Texto 1', 'Texto 3'])

    def test_cargar_textos_excel_correcto(self):
        textos = cargar_textos('archivo_correcto.xlsx')
        self.assertIsInstance(textos, list)
        self.assertEqual(len(textos), 3)
        self.assertEqual(textos, ['Texto 1', 'Texto 2', 'Texto 3'])
if __name__ == '__main__':
    unittest.main(argv=[''], exit=False) 

.....

Archivo archivo_no_existe.xlsx no encontrado: [Errno 2] No such file or directory: 'archivo_no_existe.xlsx'
Error al cargar o procesar el archivo archivo_no_excel.txt: File is not a zip file


.....

Archivo  no encontrado: [Errno 2] No such file or directory: ''
Archivo archivo_no_existe.xlsx no encontrado: [Errno 2] No such file or directory: 'archivo_no_existe.xlsx'


..
----------------------------------------------------------------------
Ran 12 tests in 0.538s

OK


Error al cargar o procesar el archivo archivo_no_excel.txt: File is not a zip file
Archivo  no encontrado: [Errno 2] No such file or directory: ''


In [4]:
class TestLiberarMemoriaGPU(unittest.TestCase):
    def test_liberar_memoria_gpu(self):
        if not torch.cuda.is_available():
            self.skipTest("CUDA no está disponible")

        # Asignar memoria en la GPU con varios tensores grandes
        tensors = torch.randn(10000, 10000, device='cuda')
        torch.cuda.synchronize()

        # Medir la memoria asignada antes
        memoria_asignada_antes = torch.cuda.memory_allocated()
        print(f"Memoria asignada antes: {memoria_asignada_antes / (1024 ** 2):.2f} MiB")

        # Liberar los tensores explícitamente
        del tensors
        liberar_memoria_gpu()
        torch.cuda.synchronize()

        # Medir la memoria asignada después
        memoria_asignada_despues = torch.cuda.memory_allocated()
        print(f"Memoria asignada después: {memoria_asignada_despues / (1024 ** 2):.2f} MiB")

        # Verificar que la memoria asignada disminuyó después de liberar la memoria
        self.assertGreater(memoria_asignada_antes, memoria_asignada_despues, "La memoria asignada no se liberó correctamente")

    def test_liberar_memoria_gpu_sin_referencia(self):
        if not torch.cuda.is_available():
            self.skipTest("CUDA no está disponible")

        # Asignar memoria en la GPU
        tensor = torch.randn(10000, 10000, device='cuda')
        torch.cuda.synchronize()

        # Medir la memoria reservada antes
        memoria_reservada_antes = torch.cuda.memory_reserved()

        # Liberar la memoria de la GPU
        del tensor
        liberar_memoria_gpu()
        torch.cuda.synchronize()
        time.sleep(1)  # Esperar un momento para asegurar que la memoria se libere

        # Medir la memoria reservada después
        memoria_reservada_despues = torch.cuda.memory_reserved()

        # Verificar que la memoria reservada disminuyó después de liberar la memoria
        self.assertGreater(memoria_reservada_antes, memoria_reservada_despues, "La memoria reservada no se liberó correctamente")

if __name__ == '__main__':
    unittest.main(argv=[''], exit=False) 

.....

Archivo archivo_no_existe.xlsx no encontrado: [Errno 2] No such file or directory: 'archivo_no_existe.xlsx'
Error al cargar o procesar el archivo archivo_no_excel.txt: File is not a zip file


......

Archivo  no encontrado: [Errno 2] No such file or directory: ''
Archivo archivo_no_existe.xlsx no encontrado: [Errno 2] No such file or directory: 'archivo_no_existe.xlsx'
Error al cargar o procesar el archivo archivo_no_excel.txt: File is not a zip file


.

Archivo  no encontrado: [Errno 2] No such file or directory: ''


.

Memoria asignada antes: 382.00 MiB
Memoria asignada después: 0.00 MiB


.
----------------------------------------------------------------------
Ran 14 tests in 2.194s

OK


In [ ]:
class TestCargarModelo(unittest.TestCase):
    def setUp(self):
        if not torch.cuda.is_available():
            self.skipTest("CUDA no está disponible")

    def tearDown(self):
        liberar_memoria_gpu()

    def test_cargar_modelo_meta_llama(self):
        model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
        try:
            model, tokenizer = cargar_modelo(model_name)
            self.assertIsNotNone(model, "El modelo no se ha cargado correctamente")
            self.assertIsNotNone(tokenizer, "El tokenizador no se ha cargado correctamente")
        except Exception as e:
            self.fail(f"cargar_modelo() lanzó una excepción inesperada: {e}")

    def test_cargar_modelo_phi_3(self):
        model_name = "microsoft/Phi-3-medium-128k-instruct"
        try:
            model, tokenizer = cargar_modelo(model_name)
            self.assertIsNotNone(model, "El modelo no se ha cargado correctamente")
            self.assertIsNotNone(tokenizer, "El tokenizador no se ha cargado correctamente")
        except Exception as e:
            self.fail(f"cargar_modelo() lanzó una excepción inesperada: {e}")

    def test_cargar_modelo_no_implementado(self):
        model_name = "modelo/no-implementado"
        with self.assertRaises(ValueError) as context:
            cargar_modelo(model_name)
        self.assertIn("El modelo \"modelo/no-implementado\" no tiene implementada su carga.", str(context.exception))

if __name__ == '__main__':
    unittest.main(argv=[''], exit=False) 

.....

Archivo archivo_no_existe.xlsx no encontrado: [Errno 2] No such file or directory: 'archivo_no_existe.xlsx'
Error al cargar o procesar el archivo archivo_no_excel.txt: File is not a zip file


.

Archivo  no encontrado: [Errno 2] No such file or directory: ''
Hora de carga:  15:37:53
Cargando meta-llama/Meta-Llama-3-8B-Instruct ...


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

..Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Hora de carga:  15:38:00
Hora de carga:  15:38:00
Cargando microsoft/Phi-3-medium-128k-instruct ...


`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attenton` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

In [ ]:
class TestGenerarRespuestas(unittest.TestCase):

    def setUp(self):
        # Configuraciones de prueba
        self.model_name_phi = "microsoft/Phi-3-medium-128k-instruct"
        self.model_name_llama = "meta-llama/Meta-Llama-3-8B-Instruct"
        self.textos = ["Esto es una prueba.", "Este es otro texto de prueba."]
        self.prompt = "Genera una respuesta:"
        self.gen_config = GenerationConfig()

        # Crear archivo de prueba
        data = {'Texto': []}
        df = pd.DataFrame(data)
        df.to_excel('resultados.xlsx', index=False)

    def tearDown(self):
        # Eliminar archivos de prueba después de cada prueba
        if os.path.exists('resultados.xlsx'):
            os.remove('resultados.xlsx')
        liberar_memoria_gpu()

    def test_generar_respuestas_phi(self):
        try:
            generar_respuestas(self.model_name_phi, self.textos, self.prompt, self.gen_config, 'resultados.xlsx')
            df = pd.read_excel('resultados.xlsx', engine='openpyxl')
            self.assertIn(self.model_name_phi, df.columns)
            self.assertEqual(len(df[self.model_name_phi]), len(self.textos))
        except Exception as e:
            self.fail(f"generar_respuestas() lanzó una excepción inesperada: {e}")

    def test_generar_respuestas_llama(self):
        try:
            generar_respuestas(self.model_name_llama, self.textos, self.prompt, self.gen_config, 'resultados.xlsx')
            df = pd.read_excel('resultados.xlsx', engine='openpyxl')
            self.assertIn(self.model_name_llama, df.columns)
            self.assertEqual(len(df[self.model_name_llama]), len(self.textos))
        except Exception as e:
            self.fail(f"generar_respuestas() lanzó una excepción inesperada: {e}")

    def test_generar_respuestas_modelo_no_implementado(self):
        model_name_invalido = "modelo/no-implementado"
        with self.assertRaises(ValueError) as context:
            generar_respuestas(model_name_invalido, self.textos, self.prompt, self.gen_config, 'resultados.xlsx')
        self.assertIn("El modelo \"modelo/no-implementado\" no tiene implementada su carga.", str(context.exception))

if __name__ == '__main__':
    unittest.main(argv=[''], exit=False) 